# Target Module Ablation — RunPod Version

This notebook runs a small supplemental target-module ablation for the PEFT Medical VQA project.

**Design:** LoRA rank=8 only, same base model / data split / epochs / learning rate / seed. Only `target_modules` changes.

Experiments:
- `attn_only`: `q_proj, k_proj, v_proj, o_proj`
- `ffn_only`: `gate_proj, up_proj, down_proj`
- `qv_only`: `q_proj, v_proj` optional

Recommended minimal run: `attn_only + ffn_only`.

In [ ]:
# Step 0: Environment setup for RunPod
import os, sys
from pathlib import Path

PROJECT_ROOT = Path('/workspace/GR5293-peft-medical-vqa')
REPO_URL = 'https://github.com/qiujunzhang03-7/GR5293-peft-medical-vqa.git'

if not PROJECT_ROOT.exists():
    !git clone {REPO_URL} {PROJECT_ROOT}

os.chdir(PROJECT_ROOT)
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

print('CWD:', os.getcwd())
!nvidia-smi

# Install dependencies and remove incompatible torchao if present.
%pip install -q -r requirements.txt
%pip uninstall -y torchao

try:
    import torchao
    print('⚠️ torchao still installed:', torchao.__version__)
except ImportError:
    print('✅ torchao not installed')

import torch
print('CUDA available:', torch.cuda.is_available())
print('GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'None')


In [ ]:
# Step 1: Write target-module ablation configs
from pathlib import Path

PROJECT_ROOT = Path('/workspace/GR5293-peft-medical-vqa')
CONFIG_DIR = PROJECT_ROOT / 'configs'
CONFIG_DIR.mkdir(parents=True, exist_ok=True)

def write_config(filename, target_modules, output_dir):
    mods = '\n'.join([f'  - {m}' for m in target_modules])
    lines = [
        f'# Target-module ablation: {filename}',
        'model_id: "Qwen/Qwen2-VL-2B-Instruct"',
        'dtype: "float16"',
        'load_in_4bit: false',
        '',
        'train_split: "train"',
        'train_max_examples: null',
        'eval_split: "test"',
        'eval_max_examples: null',
        '',
        'lora_r: 8',
        'lora_alpha: 16',
        'lora_dropout: 0.05',
        'target_modules:',
        mods,
        'use_dora: false',
        '',
        'num_epochs: 3',
        'per_device_batch_size: 1',
        'gradient_accumulation_steps: 8',
        'learning_rate: 1.0e-4',
        'weight_decay: 0.0',
        'warmup_ratio: 0.03',
        'max_grad_norm: 1.0',
        'gradient_checkpointing: true',
        '',
        'logging_steps: 25',
        f'output_dir: "{output_dir}"',
        'seed: 42',
        '',
        'run_eval_after_training: true',
        'eval_max_new_tokens: 64',
        '',
    ]
    text = '\n'.join(lines)
    path = CONFIG_DIR / filename
    path.write_text(text)
    print('wrote:', path)

write_config('lora_rank8_attn_only.yaml', ['q_proj','k_proj','v_proj','o_proj'], 'checkpoints/lora_rank8_attn_only')
write_config('lora_rank8_ffn_only.yaml', ['gate_proj','up_proj','down_proj'], 'checkpoints/lora_rank8_ffn_only')
write_config('lora_rank8_qv_only.yaml', ['q_proj','v_proj'], 'checkpoints/lora_rank8_qv_only')

!ls -lh configs/lora_rank8_*only.yaml


In [ ]:
# Step 2: Choose experiments to run
# Recommended: run attn_only + ffn_only. qv_only is optional.

EXPERIMENTS_TO_RUN = [
    'attn_only',
    'ffn_only',
    'qv_only',

config_map = {
    'attn_only': 'configs/lora_rank8_attn_only.yaml',
    'ffn_only': 'configs/lora_rank8_ffn_only.yaml',
    'qv_only': 'configs/lora_rank8_qv_only.yaml',
}

print('Experiments to run:', EXPERIMENTS_TO_RUN)
for exp in EXPERIMENTS_TO_RUN:
    print(exp, '->', config_map[exp])


In [ ]:
# Step 3: Run training with real-time logs
import subprocess, sys, time, os
from pathlib import Path

os.chdir('/workspace/GR5293-peft-medical-vqa')

def run_streaming(cmd):
    print('\n$ ' + ' '.join(cmd), flush=True)
    proc = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
    for line in proc.stdout:
        print(line, end='', flush=True)
    proc.wait()
    if proc.returncode != 0:
        raise RuntimeError(f'Command failed with return code {proc.returncode}: {cmd}')

for exp in EXPERIMENTS_TO_RUN:
    cfg = config_map[exp]
    print('\n' + '='*80)
    print(f'启动 target-module ablation: {exp} -> {cfg}')
    print('='*80)
    start = time.time()
    run_streaming([sys.executable, '-m', 'src.training.train_lora', '--config', cfg])
    print(f'✅ {exp} finished in {(time.time()-start)/3600:.2f} hours')


In [ ]:
# Step 4: Verify outputs
from pathlib import Path

expected_dirs = {
    'attn_only': Path('/workspace/GR5293-peft-medical-vqa/checkpoints/lora_rank8_attn_only'),
    'ffn_only': Path('/workspace/GR5293-peft-medical-vqa/checkpoints/lora_rank8_ffn_only'),
    'qv_only': Path('/workspace/GR5293-peft-medical-vqa/checkpoints/lora_rank8_qv_only'),
}

required = ['training_metrics.json', 'per_example_scores.json', 'lora_predictions.jsonl', 'adapter_model.safetensors', 'adapter_config.json']

for exp in EXPERIMENTS_TO_RUN:
    d = expected_dirs[exp]
    print('\n===', exp, d, '===')
    print('exists:', d.exists())
    for f in required:
        print(f, '✅' if (d/f).exists() else '❌')


In [ ]:
# Step 5: Back up checkpoints to /workspace/peft_medical_vqa_backup/checkpoints
import shutil
from pathlib import Path

src_root = Path('/workspace/GR5293-peft-medical-vqa/checkpoints')
dst_root = Path('/workspace/peft_medical_vqa_backup/checkpoints')
dst_root.mkdir(parents=True, exist_ok=True)

for exp in EXPERIMENTS_TO_RUN:
    name = {
        'attn_only': 'lora_rank8_attn_only',
        'ffn_only': 'lora_rank8_ffn_only',
        'qv_only': 'lora_rank8_qv_only',
    }[exp]
    src = src_root / name
    dst = dst_root / name
    if src.exists():
        shutil.copytree(src, dst, dirs_exist_ok=True)
        print('backed up:', dst)

print('\nBackup contents:')
!find /workspace/peft_medical_vqa_backup/checkpoints -maxdepth 2 -name "per_example_scores.json" | sort


In [ ]:
# Step 6: Create target-module ablation table and simple plot
import os, json
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

PROJECT_ROOT = Path('/workspace/GR5293-peft-medical-vqa')
os.chdir(PROJECT_ROOT)

methods_to_load = [
    ('Full attn+FFN', 'checkpoints/lora_rank8'),
    ('Attention-only', 'checkpoints/lora_rank8_attn_only'),
    ('FFN-only', 'checkpoints/lora_rank8_ffn_only'),
    ('Q/V-only', 'checkpoints/lora_rank8_qv_only'),
]

rows = []
for label, path_str in methods_to_load:
    p = PROJECT_ROOT / path_str
    scores_path = p / 'per_example_scores.json'
    metrics_path = p / 'training_metrics.json'
    if not scores_path.exists():
        print('skip missing:', label, scores_path)
        continue
    with open(scores_path) as f:
        s = json.load(f)
    metrics = None
    if metrics_path.exists():
        with open(metrics_path) as f:
            metrics = json.load(f)
    qtypes = s['qtypes']
    cl_idx = [i for i,q in enumerate(qtypes) if q == 'closed']
    op_idx = [i for i,q in enumerate(qtypes) if q == 'open']
    trainable = metrics['params']['trainable'] if metrics and 'params' in metrics else None
    peak = metrics.get('peak_gpu_memory_gb') if metrics else None
    hours = metrics['training']['total_seconds']/3600 if metrics and 'training' in metrics else None
    rows.append({
        'Setting': label,
        'Closed EM': np.mean([s['correct'][i] for i in cl_idx]),
        'Open Token-F1': np.mean([s['f1'][i] for i in op_idx]),
        'Overall EM': np.mean(s['correct']),
        'Trainable Params': trainable,
        'Peak GPU GB': peak,
        'Train Hours': hours,
    })

df = pd.DataFrame(rows)
print(df.to_string(index=False))

out_tables = PROJECT_ROOT / 'results/tables'
out_figs = PROJECT_ROOT / 'results/figures'
out_tables.mkdir(parents=True, exist_ok=True)
out_figs.mkdir(parents=True, exist_ok=True)

df.to_markdown(out_tables / 'target_module_ablation.md', index=False)
df.to_csv(out_tables / 'target_module_ablation.csv', index=False)

if not df.empty:
    ax = df.plot(x='Setting', y=['Closed EM','Open Token-F1','Overall EM'], kind='bar', figsize=(10,5))
    ax.set_ylim(0, max(0.8, df[['Closed EM','Open Token-F1','Overall EM']].max().max()+0.05))
    ax.set_ylabel('Score')
    ax.set_title('Target Module Ablation at LoRA r=8')
    plt.xticks(rotation=20, ha='right')
    plt.tight_layout()
    plt.savefig(out_figs / 'target_module_ablation.png', dpi=300, bbox_inches='tight')
    plt.savefig(out_figs / 'target_module_ablation.pdf', bbox_inches='tight')
    plt.show()

print('Saved:')
print(out_tables / 'target_module_ablation.md')
print(out_figs / 'target_module_ablation.png')


In [ ]:
# Step 7: Zip target-module ablation outputs for download
import shutil
from pathlib import Path
import zipfile

zip_path = shutil.make_archive(
    '/workspace/target_module_ablation_outputs',
    'zip',
    '/workspace/GR5293-peft-medical-vqa',
    'results'
)
print('Created:', zip_path)

ckpt_zip = '/workspace/target_module_ablation_checkpoints.zip'
folders = [
    '/workspace/GR5293-peft-medical-vqa/checkpoints/lora_rank8_attn_only',
    '/workspace/GR5293-peft-medical-vqa/checkpoints/lora_rank8_ffn_only',
    '/workspace/GR5293-peft-medical-vqa/checkpoints/lora_rank8_qv_only',
]
with zipfile.ZipFile(ckpt_zip, 'w', zipfile.ZIP_DEFLATED) as z:
    for folder in folders:
        p = Path(folder)
        if p.exists():
            for f in p.rglob('*'):
                if f.is_file():
                    z.write(f, f.relative_to('/workspace/GR5293-peft-medical-vqa/checkpoints'))
print('Created:', ckpt_zip)
